## Indicate Dynawo and Modelica package.mo

In [ ]:
#=  The env variable DYNAWO_PATH must has be set beforehands to the root of the dynawo code.
    In case you are only working with the library distribution, the second part of the expression must be slightly modified.
    Anyway the package_file should be pointing to the package.mo file of the Dynawo library you want to use
=#

# Ruta de la librería Dynawo
ENV["DYNAWO_PATH"] = ".../dynawo/ddb/Dynawo"
Dynawo_package_file = string(ENV["DYNAWO_PATH"], "/package.mo")

# Ruta de la librería Modelica (ajusta según tu instalación)
ENV["MODELICA_PATH"] = ".../dynawo/OpenModelica/lib/omlibrary/Modelica"
Modelica_package_file = string(ENV["MODELICA_PATH"], "/package.mo")

## Import packages and data

In [15]:
using OMJulia
using Plots, DataFrames, CSV

Before this step, make sure to add TripleInertialGrid.mo and StaticTripleCase.mo to the folder DYNAWO_PATH/Dynawo/Examples/InertialGrid

In [ ]:
StaticTripleCase = OMJulia.OMCSession()
ModelicaSystem(StaticTripleCase,Dynawo_package_file,"Dynawo.Examples.InertialGrid.StaticTripleCase",[Modelica_package_file])

TripleIG = OMJulia.OMCSession()
ModelicaSystem(TripleIG,Dynawo_package_file,"Dynawo.Examples.InertialGrid.TripleInertialGrid",[Modelica_package_file])

## First simulation run

The TripleInertialGrid model does not have the right initialization values, so the first seconds of the simulation are wrong.

In [17]:
buildModel(TripleIG)
resultfile_name = string("TripleIG_base", ".csv")
simulate(TripleIG,resultfile = resultfile_name,
                simflags = "-override=outputFormat=csv,startTime=0,stopTime=50")
resultfile = joinpath(getWorkDirectory(TripleIG), resultfile_name)
df = DataFrame(CSV.File(resultfile));

In [ ]:
plotlyjs()
p2 = plot(df[!, "time"], [df[!, "inertialGrid1.reducedOrderSFR.deltaFrequency"] df[!, "inertialGrid2.reducedOrderSFR.deltaFrequency"]], label=["deltaF_IG1" "deltaF_IG2"])
plot!(p2, legend=:bottomright, titlefontsize=12, labelfontsize=10)
title!(p2, "Frequency variations in the 2/3 inertial grids")
xlabel!(p2, "Time (s)")
ylabel!(p2, "deltaFrequency (Hz)")  

In [ ]:
plotlyjs()
p = plot(df[!, "time"], df[!, "deltaFrequency"], label=string("deltaF"))
plot!(p, legend=:bottomright, titlefontsize=12, labelfontsize=10)
title!(p, "Frequency difference between the 2/3 inertial grids")
xlabel!(p, "Time (s)")
ylabel!(p, "deltaFrequency (Hz)")

## Initialization using the static model

The right initialization values can be obtained from the simulation of the equivalent static model StaticTripleCase.

In [ ]:
buildModel(StaticTripleCase)
simulate(StaticTripleCase)

(U1, U2, U3, UL, UPhase1, UPhase2, UPhase3, UPhaseL, P1, Q1) = getContinuous(StaticTripleCase, ["busIG1.UPu", "busIG2.UPu", "busIG3.UPu", "busL.UPu", "busIG1.UPhase", "busIG2.UPhase", "busIG3.UPhase", "busL.UPhase", "line1.P1Pu", "line1.Q1Pu"])

In [ ]:
setParameters(TripleIG, ["inertialGrid1.H = " * "2.6", "inertialGrid2.H = " * "2.6", "inertialGrid3.H = " * "2.6"])
setParameters(TripleIG, ["deltaPPu = " * "0.02"])
setParameters(TripleIG, ["inertialGrid1.U0Pu = " * string(U1), "inertialGrid2.U0Pu = " * string(U2), "inertialGrid3.U0Pu = " * string(U3),"load.u0Pu.re = " * string(UL*cos(UPhaseL)), "inertialGrid1.UPhase0 = " * string(UPhase1), "inertialGrid2.UPhase0 = " * string(UPhase2), "inertialGrid3.UPhase0 = " * string(UPhase3),"load.u0Pu.im = " * string(UL*sin(UPhaseL)), "inertialGrid1.P0Pu = " * string(P1), "inertialGrid1.Q0Pu = " * string(Q1)])
resultfile_name = string("TripleIG_modified", ".csv")
simulate(
    TripleIG,
    resultfile = resultfile_name,
    simflags = "-override=outputFormat=csv,startTime=0,stopTime=50"
)

resultfile = joinpath(getWorkDirectory(TripleIG), resultfile_name)
df = DataFrame(CSV.File(resultfile));

In [ ]:
plotlyjs()
p2 = plot(df[!, "time"], [df[!, "inertialGrid1.reducedOrderSFR.deltaFrequency"] df[!, "inertialGrid2.reducedOrderSFR.deltaFrequency"]], label=["deltaF_IG1" "deltaF_IG2"])
plot!(p2, legend=:bottomright, titlefontsize=12, labelfontsize=10)
title!(p2, "Frequency variations in the two inertial grids")
xlabel!(p2, "Time (s)")
ylabel!(p2, "deltaFrequency (Hz)")  

In [ ]:
plotlyjs()
p = plot(df[!, "time"], df[!, "deltaFrequency"], label=string("deltaF"))
plot!(p, legend=:bottomright, titlefontsize=12, labelfontsize=10)
title!(p, "Frequency difference between the two inertial grids")
xlabel!(p, "Time (s)")
ylabel!(p, "deltaFrequency (Hz)")